# Gemma 2 2B: Unsloth sequence classification
Kaggle: **Internet Off, GPU T4 x2**. Attach the completed **gemma2b-unsloth-setting** output and Name That LLM competition test data. Training/validation uses a shared 10,000-row stratified sample of the open-pool Parquet input (seed 42).
This experiment preserves the original 18-class score head, label order, stratified splitting method, 3 epochs, LR 2e-4, LoRA rank/alpha 16 and dropout 0.05, and effective batch 64 per fold. It uses Unsloth's classifier loading and adapter preparation; Transformers Trainer and PEFT still provide underlying training components.
Two independent GPU processes train folds concurrently (GPU 0: folds 0 and 2; GPU 1: fold 1). This is fold parallelism, not distributed training of one model.
Evaluation microbatch is 8 instead of 64 for T4 memory. No text truncation is introduced: the original tokenizer did not apply its configured 2048 limit. Seed is reset for each fold, so results need not be bit-identical to the original run. Nonzero LoRA dropout preserves the baseline but limits some Unsloth optimizations.
Timing includes worker startup/compilation, training, validation and test inference; package installation is reported separately. The original train.csv Unsloth run took 3,814 seconds. The source pool has 192,926 rows. The benchmark samples 10,000 rows versus the original 2,699: a naive size-scaled estimate is 3h 56m for all 3 folds and 3 epochs on T4 x2, assuming similar per-row cost. This is an estimate, not a measured runtime. Open-pool CV measures a different validation population; its class distribution is also less balanced.
Source: [Unsloth classification loader and SEQ_CLS adapter implementation](https://github.com/unslothai/unsloth/blob/main/unsloth/models/llama.py).
Warmup uses an explicit optimizer-step count (10% rounded up), compatible with Transformers versions that removed warmup_ratio.

Unsloth auto compilation is disabled before import to bypass the observed Inductor backward assertion. This may reduce speed; the benchmark records this setting. Other Unsloth model/LoRA patches remain enabled.

Prediction uses ClassificationTrainer to bypass Unsloth's prompt-only unlabeled prediction path. Completed fold adapters are saved before inference. Every training run starts from the base model with fresh adapters.

The final results cell runs in a fresh Python process and can be run alone to recover submission.parquet from three saved fold NPZ files. It does not import sklearn or load model weights.

Training input: `/kaggle/input/datasets/crtproduction/name-that-llm-open-pool/ntl_open_pool.parquet`. A stratified sample preserves the pool class proportions; no train.csv rows are appended. The exact sample and sampling metadata are saved with the results and reused for scoring. Validation/checkpoint interval scales from 40 steps at 2,699 rows to 148 steps at 10,000 rows, avoiding a full pool validation every 40 steps. Epoch count and number of folds remain 3.


In [ ]:
import sys, os, json, time, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError, distributions
from packaging.requirements import Requirement
NOTEBOOK_START = time.perf_counter()
matches = list(Path('/kaggle/input').rglob('gemma2b-unsloth-bundle.json'))
if len(matches) != 1:
    raise RuntimeError(f'Attach exactly one completed Unsloth setting output; found {len(matches)} bundles.')
BUNDLE = matches[0].parent
manifest = json.loads(matches[0].read_text())
if manifest['python'] != list(sys.version_info[:2]):
    raise RuntimeError('Setup and training Python versions differ. Rebuild setup on this Kaggle image.')
def same_release(actual, expected):
    return actual is not None and actual.split('+')[0] == expected.split('+')[0]

def is_gpu_runtime(name):
    name = name.lower().replace('_', '-')
    return name in {'torch', 'torchvision', 'torchaudio', 'triton', 'pytorch-triton'} or name.startswith('nvidia-')

for name, expected in manifest['runtime'].items():
    try:
        actual = version(name)
    except PackageNotFoundError:
        actual = None
    if not same_release(actual, expected):
        raise RuntimeError(f'{name}: setup used {expected}, training has {actual}. Use matching Kaggle images.')
# Preserve Kaggle's installed CUDA stack. The setup may have used a CPU image.
# Keep pip dependency validation enabled: incompatible wheels must fail explicitly.
installed_runtime = {}
for dist in distributions():
    name = dist.metadata.get('Name', '')
    if is_gpu_runtime(name):
        installed_runtime[name.lower().replace('_', '-')] = dist.version
constraints = Path('/kaggle/working/installed-gpu-constraints.txt')
constraints.write_text(''.join(f'{name}=={ver}\n' for name, ver in sorted(installed_runtime.items())))
requirements = []
for line in (BUNDLE/'offline-requirements.txt').read_text().splitlines():
    if line.strip() and not line.lstrip().startswith('#'):
        if not is_gpu_runtime(Requirement(line).name):
            requirements.append(line)
install_requirements = Path('/kaggle/working/unsloth-install-requirements.txt')
install_requirements.write_text('\n'.join(requirements) + '\n')
print('Preserving installed GPU packages:', installed_runtime)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    '--find-links', str(BUNDLE/'wheels'),
                    '--constraint', str(constraints), '-r', str(install_requirements)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        'Offline dependencies could not be resolved against the installed GPU stack. '
        'Keep the pip error above. Rebuild setup with GPU T4 x2 and Internet On '
        'on the same Kaggle image, then attach that completed version.'
    ) from exc
INSTALL_SECONDS = time.perf_counter() - NOTEBOOK_START
print('Offline bundle:', BUNDLE)

In [ ]:
DATA = Path('/kaggle/input/competitions/name-that-llm')
POOL_FILE = Path('/kaggle/input/datasets/crtproduction/name-that-llm-open-pool/ntl_open_pool.parquet')
if not POOL_FILE.is_file():
    raise FileNotFoundError(f'Attach the open-pool dataset: {POOL_FILE}')
if not (DATA/'test.csv').is_file():
    raise FileNotFoundError('Attach competition test.csv at DATA.')
OUTPUT = Path('/kaggle/working/gemma2b-unsloth-results')
OUTPUT.mkdir(exist_ok=True)
WORKERS = 2  # Set to 1 only for a single-GPU experiment.

MAX_TRAIN_ROWS = 10_000
SAMPLE_SEED = 42
TRAIN_FILE = OUTPUT/'sampled-training.parquet'
# Prepare one shared sample in a fresh interpreter after dependency installation.
SAMPLE_SOURCE = r'''
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
pool_path, dest_path, limit, seed = sys.argv[1:]
limit, seed = int(limit), int(seed)
if limit <= 0:
    raise ValueError('MAX_TRAIN_ROWS must be positive.')
frame = pd.read_parquet(pool_path, columns=['id', 'text', 'model'])
if frame[['id', 'text', 'model']].isna().any().any() or frame['id'].duplicated().any():
    raise ValueError('Pool IDs must be unique; id, text and model cannot be missing.')
source_counts = frame['model'].value_counts().sort_index()
if limit < len(frame):
    indices, _ = train_test_split(np.arange(len(frame)), train_size=limit,
                                 stratify=frame['model'], random_state=seed)
    sampled = frame.iloc[indices].reset_index(drop=True)
else:
    sampled = frame.reset_index(drop=True)
counts = sampled['model'].value_counts().sort_index()
if set(counts.index) != set(source_counts.index) or counts.min() < 3:
    raise ValueError('Sample must retain every model with at least three rows for CV.')
dest = Path(dest_path)
sampled.to_parquet(dest, index=False)
metadata = {'source':pool_path, 'source_rows':len(frame), 'sample_rows':len(sampled),
            'requested_rows':limit, 'seed':seed, 'method':'stratified by model, proportional allocation',
            'class_counts':{str(k):int(v) for k,v in counts.items()}}
(dest.parent/'sampling.json').write_text(json.dumps(metadata, indent=2))
print(f'Saved stratified sample: {len(sampled):,} rows, {len(counts)} model labels, seed {seed}')
print(counts.to_string())
'''
subprocess.run([sys.executable, '-c', SAMPLE_SOURCE, str(POOL_FILE), str(TRAIN_FILE),
                str(MAX_TRAIN_ROWS), str(SAMPLE_SEED)], check=True)


In [ ]:
WORKER_SOURCE = r'''


import os
# Avoid the Inductor backward assertion seen on Kaggle Torch 2.11 / T4.
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
import argparse, json, time, math
p = argparse.ArgumentParser()
p.add_argument('--bundle', required=True)
p.add_argument('--data', required=True)
p.add_argument('--train-file', required=True)
p.add_argument('--output', required=True)
p.add_argument('--folds', required=True)
a = p.parse_args()
started = time.perf_counter()
from unsloth import FastLanguageModel
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import torch
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

class CFG:
    VER = 1
    AUTHOR = "takaito"
    COMPETITION = "NameThatLLM"
    DATA_PATH = Path("/kaggle/input/competitions/name-that-llm")
    MODEL_PATH = "/kaggle/input/notebooks/takaito/name-that-llm-gemma2b-setting/gemma-2-2b-it-bnb-4bit"
    MODEL_NAME = "gemma-2-2b-it-bnb-4bit"
    MAX_LENGTH = 2048
    EPOCHS = 3
    STEPS = 40
    WARMUP_RATIO = 0.1
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-4
    OPTIM = "adamw_8bit"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 3
    TARGET_COL = "model"
    TARGET_COL_CLASS_NUM = 18
    METRIC = "acc"
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
CFG.MODEL_PATH = str(Path(a.bundle) / 'gemma-2-2b-it-bnb-4bit')
CFG.DATA_PATH = Path(a.data)
CFG.AUTHOR = 'zuhairsan'
CFG.PREFIX = str(Path(a.output) / 'gemma2b-unsloth')
# Smaller evaluation microbatches reduce T4 memory use without changing probabilities.
CFG.EVAL_BATCH_SIZE = 8
if not torch.cuda.is_available():
    raise RuntimeError('Select GPU T4 x2 in Kaggle.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
TARGET_MODELS = [
    # OpenAI
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    # Google
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    # Anthropic
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    # Qwen
    "qwen3.8-max",
    # ZAI
    "glm-5.3",
    "glm-5.3-flash",
    # Moonshot AI
    "kimi-k3",
]

model2id = {m: i for i, m in enumerate(TARGET_MODELS)}
id2model = {i: m for m, i in model2id.items()}
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.use_deterministic_algorithms(True)
train_df = pd.read_parquet(a.train_file, columns=['id', 'text', 'model'])
if train_df[['id', 'text', 'model']].isna().any().any():
    raise ValueError('Open-pool data contains missing id, text or model values.')
if train_df['model'].value_counts().min() < CFG.N_SPLIT:
    raise ValueError('Each class needs at least N_SPLIT rows.')
# Keep validation/checkpoint frequency proportional to the original 2,699-row run.
CFG.STEPS = max(40, round(40 * len(train_df) / 2699))
print(f'Training pool: {len(train_df):,} rows; evaluation/save interval: {CFG.STEPS} steps', flush=True)
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")
train_df["labels"] = train_df[CFG.TARGET_COL].map(model2id)
if train_df['labels'].isna().any():
    raise ValueError('Unknown target model label in open-pool Parquet.')
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH, local_files_only=True)
tokenizer.add_eos_token = True
tokenizer.padding_side = "right"

def tokenize(example, tokenizer):
    return tokenizer(example["text"], add_special_tokens=True)
def mk_dataset(df, tokenizer):
    raw_datasets = Dataset.from_pandas(df)
    tokenized_datasets = raw_datasets.map(
        tokenize, 
        fn_kwargs={"tokenizer": tokenizer}
    ).remove_columns(["text"])
    return tokenized_datasets
ds = mk_dataset(train_df[["text", "labels"]], tokenizer)
ds_test = mk_dataset(test_df[["text"]], tokenizer)
# The baseline declares MAX_LENGTH=2048 but does not truncate. Preserve every token.
CFG.MAX_LENGTH = max(CFG.MAX_LENGTH, max(map(len, ds['input_ids'])),
                     max(map(len, ds_test['input_ids']), default=0))
print('Context capacity:', CFG.MAX_LENGTH, flush=True)
def compute_metrics(p):
    label_preds, labels = p
    label_prob_preds = torch.softmax(torch.tensor(label_preds), dim = 1).numpy()
    score = accuracy_score(labels, label_prob_preds.argmax(axis=1))
    return {"acc": score}
def get_args(CFG, fold, n_train):
    # Each worker has one GPU; count optimizer updates including a final partial batch.
    microbatch = CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT
    batches_per_epoch = math.ceil(n_train / microbatch)
    updates_per_epoch = math.ceil(batches_per_epoch / CFG.TRAIN_BATCH_SPLIT)
    total_updates = math.ceil(updates_per_epoch * CFG.EPOCHS)
    warmup_steps = math.ceil(CFG.WARMUP_RATIO * total_updates)
    print(f'Fold {fold}: {total_updates} optimizer steps, {warmup_steps} warmup steps', flush=True)
    args = TrainingArguments(
        output_dir=f"{CFG.PREFIX}-fold{fold}",
        report_to="none",
        eval_strategy="steps",
        logging_strategy="steps",
        save_strategy="steps",
        eval_steps=CFG.STEPS,
        logging_steps=CFG.STEPS,
        save_steps=CFG.STEPS,
        save_total_limit=1,
        metric_for_best_model=CFG.METRIC,
        greater_is_better=CFG.METRIC_MAXIMIZE_FLAG,
        optim=CFG.OPTIM,
        learning_rate=CFG.LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_steps=warmup_steps,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE // CFG.TRAIN_BATCH_SPLIT,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.TRAIN_BATCH_SPLIT,
        num_train_epochs=CFG.EPOCHS,
        fp16=True,
        bf16=False,
        label_names=["labels"],
        seed=CFG.SEED,
        data_seed=CFG.SEED,
    )
    return args
def setting_model(CFG):
    model, _ = FastLanguageModel.from_pretrained(
        model_name=CFG.MODEL_PATH,
        max_seq_length=CFG.MAX_LENGTH,
        dtype=torch.float16,
        load_in_4bit=True,
        num_labels=CFG.TARGET_COL_CLASS_NUM,
        device_map={'': 0},
        local_files_only=True,
    )
    if 'SequenceClassification' not in type(model).__name__:
        raise RuntimeError('This Unsloth version did not load a sequence classifier.')
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.config.id2label = id2model
    model.config.label2id = model2id
    model.config.problem_type = 'single_label_classification'
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth', random_state=CFG.SEED,
    )
    # Unsloth detects SEQ_CLS; PEFT automatically trains/saves the score head.
    if not any('score' in name and param.requires_grad for name,param in model.named_parameters()):
        raise RuntimeError('The classification head is frozen; stop before training.')
    model.print_trainable_parameters()
    return model


class ClassificationTrainer(Trainer):
    # Unsloth's global prediction patch expects a prompt on unlabeled batches.
    # Tokenized sequence classification needs the same forward pass with/without labels.
    @torch.no_grad()
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get('labels')
        previous = os.environ.get('UNSLOTH_RETURN_LOGITS')
        os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
        try:
            with self.compute_loss_context_manager():
                outputs = model(**inputs, return_dict=True)
        finally:
            if previous is None:
                os.environ.pop('UNSLOTH_RETURN_LOGITS', None)
            else:
                os.environ['UNSLOTH_RETURN_LOGITS'] = previous
        loss = outputs.loss.detach().mean() if outputs.loss is not None else None
        if prediction_loss_only:
            return loss, None, None
        logits = outputs.logits.detach()
        if logits.ndim != 2 or logits.shape[-1] != CFG.TARGET_COL_CLASS_NUM:
            raise RuntimeError(f'Expected batch x 18 classification logits, got {logits.shape}')
        return loss, logits, labels.detach() if labels is not None else None


kfold = StratifiedKFold(n_splits=CFG.N_SPLIT, shuffle=True, random_state=CFG.SEED)
selected = set(map(int, a.folds.split(',')))
for fold, (train_index, valid_index) in enumerate(kfold.split(train_df, train_df[CFG.TARGET_COL])):
    if fold not in selected:
        continue
    seed_everything(CFG.SEED)
    torch.cuda.synchronize()
    fold_start = time.perf_counter()
    training_args = get_args(CFG, fold, len(train_index))
    model = setting_model(CFG)
    trainer = ClassificationTrainer(
        args=training_args, model=model,
        train_dataset=ds.select(train_index), eval_dataset=ds.select(valid_index),
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )
    trainer.train()
    # Save the completed classifier before prediction so an inference error cannot lose it.
    trainer.save_model(str(Path(a.output) / f'fold-{fold}-final'))
    train_end = time.perf_counter()
    valid_logits = trainer.predict(ds.select(valid_index)).predictions
    test_logits = trainer.predict(ds_test).predictions
    valid_prob = torch.softmax(torch.as_tensor(valid_logits), dim=1).numpy()
    test_prob = torch.softmax(torch.as_tensor(test_logits), dim=1).numpy()
    torch.cuda.synchronize()
    end = time.perf_counter()
    np.savez_compressed(Path(a.output)/f'fold-{fold}.npz', valid_index=valid_index, valid_ids=train_df.iloc[valid_index]['id'].to_numpy(dtype=str),
                        valid_prob=valid_prob, test_prob=test_prob)
    timing = {'fold':fold, 'training_file':a.train_file, 'training_rows':len(train_df), 'load_train_seconds':train_end-fold_start,
              'inference_seconds':end-train_end, 'total_seconds':end-fold_start,
              'accuracy':accuracy_score(train_df.iloc[valid_index]['labels'], valid_prob.argmax(axis=1))}
    (Path(a.output)/f'fold-{fold}-timing.json').write_text(json.dumps(timing, indent=2))
    print(timing, flush=True)
    del trainer, model, valid_logits, test_logits
    gc.collect()
    torch.cuda.empty_cache()
'''

WORKER_FILE = Path('/kaggle/working/train_gemma2b_unsloth.py')
WORKER_FILE.write_text(WORKER_SOURCE)


In [ ]:
# Each process sees one T4, avoiding DataParallel on a quantized model.
def worker_error(gpu, returncode):
    log_path = OUTPUT / f'gpu-{gpu}.log'
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-100:])
    print(f'GPU {gpu} worker traceback:\n{tail}', flush=True)
    return RuntimeError(f'GPU {gpu} exited with code {returncode}. Full log: {log_path}')

RUN_START = time.perf_counter()
assignments = ['0,2', '1'] if WORKERS == 2 else ['0,1,2']
if WORKERS not in (1,2):
    raise ValueError('WORKERS must be 1 or 2.')
processes, logs = [], []
try:
    for gpu, folds in enumerate(assignments):
        env = os.environ.copy()
        env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
                   HF_HUB_OFFLINE='1', HF_DATASETS_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
                   TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2',
                   UNSLOTH_COMPILE_LOCATION=str(OUTPUT/f'compile-gpu-{gpu}'),
                   TORCHINDUCTOR_CACHE_DIR=str(OUTPUT/f'inductor-gpu-{gpu}'))
        log = (OUTPUT/f'gpu-{gpu}.log').open('w')
        logs.append(log)
        processes.append(subprocess.Popen([sys.executable, str(WORKER_FILE),
            '--bundle', str(BUNDLE), '--data', str(DATA), '--train-file', str(TRAIN_FILE), '--output', str(OUTPUT), '--folds', folds],
            env=env, stdout=log, stderr=subprocess.STDOUT))
    while any(p.poll() is None for p in processes):
        for gpu,p in enumerate(processes):
            if p.poll() not in (None,0):
                raise worker_error(gpu, p.returncode)
        time.sleep(30)
        for gpu in range(len(processes)):
            tail = (OUTPUT/f'gpu-{gpu}.log').read_text(errors='replace').splitlines()
            print(f'GPU {gpu}: ' + (tail[-1] if tail else 'starting'), flush=True)
    for gpu, p in enumerate(processes):
        if p.returncode != 0:
            raise worker_error(gpu, p.returncode)
finally:
    for p in processes:
        if p.poll() is None:
            p.terminate()
            p.wait()
    for log in logs:
        log.close()
RUN_SECONDS = time.perf_counter() - RUN_START
print(f'Training + inference wall time: {RUN_SECONDS/60:.1f} minutes')
run_metrics = {'train_inference_wall_seconds':RUN_SECONDS, 'install_seconds':INSTALL_SECONDS,
               'workers':WORKERS, 'unsloth_compile_disabled':True, 'packages':manifest['packages'],
               'training_file':str(TRAIN_FILE)}
(OUTPUT/'run-metrics.json').write_text(json.dumps(run_metrics, indent=2))


In [ ]:
# Safe to run this cell alone: no training or model loading.
# A fresh interpreter avoids stale NumPy/SciPy binaries cached by the notebook kernel.
import sys, json, time, subprocess
from pathlib import Path
RESULTS_DIR = None  # Optionally set to an attached failed run's gemma2b-unsloth-results folder.
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
DEST_DIR = Path('/kaggle/working')
def complete_results(path):
    return all((path/f'fold-{fold}.npz').is_file() for fold in range(3))
if RESULTS_DIR is None:
    local = Path('/kaggle/working/gemma2b-unsloth-results')
    if complete_results(local):
        RESULTS_DIR = local
    else:
        candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('fold-0.npz')
                             if complete_results(p.parent)})
        if len(candidates) != 1:
            raise RuntimeError('Set RESULTS_DIR to the saved folder containing fold-0.npz, fold-1.npz and fold-2.npz. No retraining needed.')
        RESULTS_DIR = candidates[0]
RESULTS_DIR = Path(RESULTS_DIR)
TRAIN_FILE = RESULTS_DIR/'sampled-training.parquet'
if not TRAIN_FILE.is_file():
    raise FileNotFoundError('Attach the sample run output including sampled-training.parquet.')
if not complete_results(RESULTS_DIR):
    raise FileNotFoundError('All three saved fold prediction files are required.')
run_file = RESULTS_DIR/'run-metrics.json'
run = json.loads(run_file.read_text()) if run_file.is_file() else {}
if not run and 'RUN_SECONDS' in globals():
    run = {'train_inference_wall_seconds':RUN_SECONDS,
           'install_seconds':globals().get('INSTALL_SECONDS'),
           'workers':globals().get('WORKERS'), 'unsloth_compile_disabled':True}

AGGREGATE_SOURCE = r'''
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
cfg = json.loads(sys.argv[1])
DATA, RESULTS, DEST = (Path(cfg[k]) for k in ('data', 'results', 'dest'))
DEST.mkdir(parents=True, exist_ok=True)
TARGET_MODELS = [
    # OpenAI
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    # Google
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    # Anthropic
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    # Qwen
    "qwen3.8-max",
    # ZAI
    "glm-5.3",
    "glm-5.3-flash",
    # Moonshot AI
    "kimi-k3",
]

model2id = {m: i for i, m in enumerate(TARGET_MODELS)}
id2model = {i: m for m, i in model2id.items()}

train_df = pd.read_parquet(cfg['train_file'], columns=['id', 'model'])
test_df = pd.read_csv(DATA/'test.csv')
n_classes = len(TARGET_MODELS)
oof = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_probs = np.zeros((len(test_df), n_classes), dtype=np.float32)
coverage = np.zeros(len(train_df), dtype=int)
for fold in range(3):
    with np.load(RESULTS/f'fold-{fold}.npz', allow_pickle=False) as result:
        indices, valid, test = result['valid_index'], result['valid_prob'], result['test_prob']
        if indices.ndim != 1 or not np.issubdtype(indices.dtype, np.integer):
            raise ValueError(f'Fold {fold}: invalid validation indices')
        if np.any(indices < 0) or np.any(indices >= len(train_df)):
            raise ValueError(f'Fold {fold}: validation indices outside the open-pool dataset')
        if valid.shape != (len(indices), n_classes) or test.shape != test_probs.shape:
            raise ValueError(f'Fold {fold}: probability shapes do not match train/test files')
        if not np.isfinite(valid).all() or not np.isfinite(test).all():
            raise ValueError(f'Fold {fold}: non-finite probabilities')
        if 'valid_ids' not in result.files or not np.array_equal(result['valid_ids'], train_df.iloc[indices]['id'].to_numpy(dtype=str)):
            raise ValueError(f'Fold {fold}: saved predictions do not match the sampled row IDs')
        oof[indices] = valid
        np.add.at(coverage, indices, 1)
        test_probs += test / 3
if not np.all(coverage == 1):
    raise ValueError('Invalid out-of-fold coverage: every train row must appear exactly once')
# Plain classification accuracy requires no sklearn/SciPy import.
pred_labels = np.asarray(TARGET_MODELS)[oof.argmax(axis=1)]
score = float(np.mean(train_df['model'].to_numpy() == pred_labels))
submission = pd.DataFrame({'id':test_df['id'], 'model':np.asarray(TARGET_MODELS)[test_probs.argmax(axis=1)]})
submission.to_parquet(DEST/'submission.parquet', index=False)
np.save(DEST/'oof-probabilities.npy', oof)
np.save(DEST/'test-probabilities.npy', test_probs)
fold_timings = []
for fold in range(3):
    path = RESULTS/f'fold-{fold}-timing.json'
    fold_timings.append(json.loads(path.read_text()) if path.is_file() else {'fold':fold})
run = cfg.get('run', {})
seconds = run.get('train_inference_wall_seconds')
sampling = json.loads((RESULTS/'sampling.json').read_text())
summary = dict(run, sampling=sampling, cv_accuracy=score, fold_timings=fold_timings,
               training_file=cfg['train_file'], training_rows=len(train_df),
               reference_train_csv_seconds=3814, reference_train_csv_rows=2699)
(DEST/'benchmark.json').write_text(json.dumps(summary, indent=2))
print(f'CV accuracy: {score:.5f}')
if seconds:
    print(f'Training + inference wall time: {seconds/60:.1f} minutes')
print('Open-pool CV is not directly comparable to train.csv CV or the Kaggle leaderboard score.')
print(f'Saved {DEST / "submission.parquet"}')
print(f'Saved {DEST / "benchmark.json"}')
'''

job = {'data':str(DATA_DIR), 'train_file':str(TRAIN_FILE), 'results':str(RESULTS_DIR), 'dest':str(DEST_DIR), 'run':run}
subprocess.run([sys.executable, '-c', AGGREGATE_SOURCE, json.dumps(job)], check=True)
